# Разбиение данных

Вход: `data/processed/`. Выход: `data/splits/seed_42/<dataset>/`.

- **Amazon:** случайное разбиение buy-пар 80/10/10, `SEED = 42`. Обратные пары вместе;
  validation/test исключаются из обоих отношений train.
- **Retailrocket и REES46:** глобальное временное разбиение 80/10/10.
  Сессии и транзакции на границах исключаются целиком; повторные связи в разные периоды допустимы.
- Train содержит buy/view; validation/test — только buy известных товаров.
  `known_user` отмечает известных пользователей для оценки 4D.

Доли после фильтрации могут отличаться от заданных. Seed влияет только на случайный split Amazon.

In [1]:
from importlib.util import find_spec

if any(find_spec(name) is None for name in ("pandas", "numpy", "pyarrow", "duckdb")):
    %pip install -q pandas numpy pyarrow duckdb

In [2]:
import gzip
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
from pyarrow import csv as arrow_csv


ROOT = Path(".")
DATA = ROOT / "data" / "processed"
SEED = 42
TRAIN_SHARE, VALID_SHARE = 0.8, 0.1
OUT = ROOT / "data" / "splits" / f"seed_{SEED}"
CHUNK_SIZE = 250_000
SPLITS = ["train", "validation", "test"]

## Общие функции

In [3]:
def pair_keys(frame):
    """Возвращает одинаковый ключ для пары A–B и обратной B–A."""
    left, right = frame.source_id, frame.target_id
    return pd.MultiIndex.from_arrays([left.where(left <= right, right), right.where(left <= right, left)])


def split_amazon():
    """Делит Amazon по buy-парам и исключает holdout из обучающих buy/view."""
    folder = OUT / "amazon"
    folder.mkdir(parents=True, exist_ok=True)
    pairs = pd.read_csv(DATA / "amazon" / "pairs.csv.gz", dtype="string[pyarrow]")

    buy = pairs[pairs.relation.eq("buy")]
    keys = pair_keys(buy).unique().sort_values()
    order = np.random.default_rng(SEED).permutation(len(keys))

    first, second = int(len(keys) * TRAIN_SHARE), int(len(keys) * (TRAIN_SHARE + VALID_SHARE))
    valid_keys, test_keys = keys[order[first:second]], keys[order[second:]]
    holdout = valid_keys.append(test_keys)
    train = pairs[~pair_keys(pairs).isin(holdout)]
    known_items = pd.Index(pd.concat([train.source_id, train.target_id]).unique())

    frames = {"train": train, "validation": buy[pair_keys(buy).isin(valid_keys)],
              "test": buy[pair_keys(buy).isin(test_keys)]}
    for name, frame in frames.items():
        if name != "train":
            frame = frame[frame.source_id.isin(known_items) & frame.target_id.isin(known_items)]
        frame.to_csv(folder / f"{name}.csv.gz", index=False)
        print(name, len(frame))

    holdout.to_frame(index=False, name=["source_id", "target_id"]).to_csv(folder / "holdout_pairs.csv.gz", index=False)
    pd.DataFrame({"item_id": known_items.sort_values()}).to_csv(folder / "train_items.csv.gz", index=False)


In [4]:
def temporal_groups(dataset):
    """Назначает временные части событиям, исключая целиком пограничные и связанные группы."""
    columns = ["timestamp", "user_id", "item_id", "session_id"]
    if dataset == "retailrocket":
        columns.append("transactionid")
    
    events = pd.read_csv(DATA / dataset / "events.csv.gz", usecols=columns, dtype="string[pyarrow]")
    events["timestamp"] = pd.to_datetime(events.timestamp, format="ISO8601", utc=True, errors="coerce")
    invalid = events[["timestamp", "user_id", "item_id", "session_id"]].isna().any(axis=1)
    times = events.loc[~invalid, "timestamp"].sort_values().reset_index(drop=True)

    cut1 = times.iloc[int(len(times) * TRAIN_SHARE)]
    cut2 = times.iloc[int(len(times) * (TRAIN_SHARE + VALID_SHARE))]
    if cut1 >= cut2:
        raise ValueError("Временные границы совпали; выберите другой период или доли")
    
    events["split"] = (events.timestamp.ge(cut1).astype(int) + events.timestamp.ge(cut2).astype(int))
    events.loc[invalid, "split"] = -1
    group_columns = ["session_id"] + (["transactionid"] if dataset == "retailrocket" else [])
    while True:
        previous = int(events.split.eq(-1).sum())
        for column in group_columns:
            grouped = events.groupby(["user_id", column])["split"]
            low, high = grouped.transform("min"), grouped.transform("max")
            bad = low.notna() & (low.lt(0) | low.ne(high))
            events.loc[bad, "split"] = -1

        if int(events.split.eq(-1).sum()) == previous:
            break
    kept = events[events.split.ge(0)].copy()
    if set(kept.split.unique()) != {0, 1, 2}:
        raise ValueError("После исключения групп одна из временных частей пуста")
    
    mappings = {}
    for relation, column in [("view", "session_id"),
                              ("buy", "transactionid" if dataset == "retailrocket" else "session_id")]:
        groups = kept.dropna(subset=[column])[["user_id", column, "split"]].drop_duplicates()
        mappings[relation] = groups.rename(columns={column: "group_id"}).assign(relation=relation)
    train = kept[kept.split.eq(0)]
    metadata = {"validation_start": str(cut1), "test_start": str(cut2),
                "events_removed": int(events.split.eq(-1).sum())}
    return pd.concat(mappings.values(), ignore_index=True), set(train.item_id), set(train.user_id), metadata


def split_events(dataset):
    """Сохраняет пары по временным частям; для оценки оставляет покупки известных товаров."""
    folder = OUT / dataset
    folder.mkdir(parents=True, exist_ok=True)

    groups, known_items, known_users, metadata = temporal_groups(dataset)
    
    keys = ["user_id", "group_id", "relation"]
    membership = groups.set_index(keys)["split"]
    phases = membership.to_numpy()
    columns = pd.read_csv(DATA / dataset / "pairs.csv.gz", nrows=0).columns.tolist() + ["known_user"]
    counts = dict.fromkeys(SPLITS, 0)
    for name in SPLITS:
        pd.DataFrame(columns=columns).to_csv(folder / f"{name}.csv.gz", index=False)

    for chunk in pd.read_csv(DATA / dataset / "pairs.csv.gz", dtype="string[pyarrow]", chunksize=CHUNK_SIZE):
        positions = membership.index.get_indexer(pd.MultiIndex.from_frame(chunk[keys]))
        assigned = positions >= 0
        matched = chunk.loc[assigned]
        splits = phases[positions[assigned]]
        for i, name in enumerate(SPLITS):
            part = matched.loc[splits == i].copy()
            if i > 0:
                part = part[part.relation.eq("buy") & part.source_id.isin(known_items)
                            & part.target_id.isin(known_items)].copy()
            if part.empty:
                continue
            part["known_user"] = part.user_id.isin(known_users).map({True: "True", False: "False"})
            table = pa.Table.from_pandas(part, preserve_index=False)
            with gzip.open(folder / f"{name}.csv.gz", mode="ab", compresslevel=1) as stream:
                arrow_csv.write_csv(table, stream, write_options=arrow_csv.WriteOptions(include_header=False))
            counts[name] += len(part)

    pd.DataFrame({"item_id": sorted(known_items)}).to_csv(folder / "train_items.csv.gz", index=False)
    
    metadata.update(seed=SEED, train_share=TRAIN_SHARE, validation_share=VALID_SHARE)
    (folder / "split_info.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))

    print(dataset, counts)
    print("Границы:", metadata["validation_start"], metadata["test_start"])



def split_rees(dataset):
    """Делит события REES46 по целым сессиям и записывает пары через DuckDB."""
    from tempfile import TemporaryDirectory

    import duckdb

    folder = OUT / dataset
    folder.mkdir(parents=True, exist_ok=True)

    def sql_path(path):
        return "'" + str(path).replace("'", "''") + "'"

    pairs_path = sql_path(DATA / dataset / "pairs.csv.gz")
    events_path = sql_path(DATA / dataset / "events.csv.gz")
    with TemporaryDirectory(dir=folder) as temporary, duckdb.connect() as db:
        temporary = Path(temporary)
        db.execute("SET memory_limit='3GB'")
        db.execute("SET TimeZone='UTC'")
        db.execute("SET threads=4")
        db.execute("SET preserve_insertion_order=false")
        db.execute(f"SET temp_directory={sql_path(temporary / 'spill')}")
        db.execute(f"""
            CREATE TEMP TABLE events AS
            SELECT try_cast(timestamp AS TIMESTAMPTZ) AS ts, user_id, item_id, session_id
            FROM read_csv({events_path}, all_varchar=true)
        """)
        valid = "ts IS NOT NULL AND user_id IS NOT NULL AND item_id IS NOT NULL AND session_id IS NOT NULL"
        total = db.execute(f"SELECT count(*) FROM events WHERE {valid}").fetchone()[0]
        if not total:
            raise ValueError("Нет событий с корректным временем и ID")
        cutoffs = [
            db.execute(f"SELECT ts FROM events WHERE {valid} ORDER BY ts LIMIT 1 OFFSET ?", [int(total * share)]).fetchone()[0]
            for share in (TRAIN_SHARE, TRAIN_SHARE + VALID_SHARE)
        ]
        if cutoffs[0] >= cutoffs[1]:
            raise ValueError("Временные границы совпали; выберите другой период или доли")
        db.execute("""
            CREATE TEMP TABLE sessions AS
            WITH labeled AS (
                SELECT user_id, session_id,
                    CASE WHEN ts IS NULL OR item_id IS NULL THEN -1
                         WHEN ts < ? THEN 0 WHEN ts < ? THEN 1 ELSE 2 END AS phase
                FROM events
                WHERE user_id IS NOT NULL AND session_id IS NOT NULL
            )
            SELECT user_id, session_id, min(phase) AS phase, count(*) AS event_count
            FROM labeled
            GROUP BY user_id, session_id
            HAVING min(phase) >= 0 AND min(phase) = max(phase)
        """, cutoffs)
        stats = db.execute("SELECT count(DISTINCT phase), sum(event_count) FROM sessions").fetchone()
        if stats[0] != len(SPLITS):
            raise ValueError("После исключения групп одна из временных частей пуста")
        db.execute("""
            CREATE TEMP TABLE train_items AS
            SELECT DISTINCT item_id FROM events JOIN sessions USING (user_id, session_id)
            WHERE phase = 0
        """)
        db.execute("CREATE TEMP TABLE train_users AS SELECT DISTINCT user_id FROM sessions WHERE phase = 0")
        events_removed = db.execute("SELECT count(*) FROM events").fetchone()[0] - stats[1]
        parquet = sql_path(temporary / "split-pairs.parquet")
        db.execute(f"""
            COPY (
                SELECT p.*, s.phase,
                       CASE WHEN u.user_id IS NULL THEN 'False' ELSE 'True' END AS known_user
                FROM read_csv({pairs_path}, all_varchar=true) p
                JOIN sessions s ON p.user_id=s.user_id AND p.group_id=s.session_id
                LEFT JOIN train_users u ON p.user_id=u.user_id
                WHERE p.relation IN ('buy', 'view')
                  AND (s.phase=0 OR (
                    p.relation='buy' AND p.source_id IN (SELECT item_id FROM train_items)
                                     AND p.target_id IN (SELECT item_id FROM train_items)
                  ))
            ) TO {parquet} (FORMAT PARQUET)
        """)
        sizes = dict(db.execute(f"SELECT phase, count(*) FROM read_parquet({parquet}) GROUP BY phase").fetchall())
        if set(sizes) != set(range(len(SPLITS))):
            raise ValueError("После фильтрации пар одна из частей пуста")
        for i, name in enumerate(SPLITS):
            output_path = sql_path(folder / f"{name}.csv.gz")
            db.execute(f"""
                COPY (SELECT * EXCLUDE(phase) FROM read_parquet({parquet}) WHERE phase={i})
                TO {output_path} (HEADER, COMPRESSION GZIP)
            """)
        output_path = sql_path(folder / "train_items.csv.gz")
        db.execute(f"COPY (SELECT item_id FROM train_items ORDER BY item_id) TO {output_path} (HEADER, COMPRESSION GZIP)")

    metadata = {
        "validation_start": str(cutoffs[0]),
        "test_start": str(cutoffs[1]),
        "events_removed": events_removed,
        "seed": SEED,
        "train_share": TRAIN_SHARE,
        "validation_share": VALID_SHARE,
    }
    (folder / "split_info.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
    print(dataset, {name: sizes[i] for i, name in enumerate(SPLITS)})
    print("Границы:", metadata["validation_start"], metadata["test_start"])

## Amazon Musical Instruments

In [5]:
split_amazon()

train 312439


validation 8195


test 8243


## Retailrocket

In [6]:
split_events("retailrocket")

retailrocket {'train': 1020679, 'validation': 974, 'test': 758}
Границы: 2015-08-18 04:23:20.445000+00:00 2015-09-02 17:49:14.032000+00:00


## REES46 Cosmetics Events

In [7]:
split_rees("rees46_cosmetics")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rees46_cosmetics {'train': 26993228, 'validation': 1121086, 'test': 1015086}
Границы: 2020-01-31 23:40:01+00:00 2020-02-14 20:06:22+00:00


## REES46 Electronics Events

In [8]:
split_rees("rees46_electronics")

rees46_electronics {'train': 322033, 'validation': 926, 'test': 740}
Границы: 2021-01-30 10:17:09+00:00 2021-02-13 11:50:32+00:00


## Результат

Для каждого набора сохраняются `train.csv.gz`, `validation.csv.gz`, `test.csv.gz` и `train_items.csv.gz`.

Дополнительно:
- Amazon: `holdout_pairs.csv.gz` со списком отложенных пар.
- Retailrocket и REES46: `split_info.json` с границами времени, параметрами и числом исключённых событий.
